In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [8]:
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)


prompt = ChatPromptTemplate.from_template("""
You are BajajBot, an AI assistant for Bajaj Finance helpdesk agents.

CONTEXT (retrieved from Bajaj Finance policy documents):
-------------------------------------------------------
{context}
-------------------------------------------------------

INSTRUCTIONS:
- Answer ONLY using the CONTEXT above.
- Do NOT use your training knowledge.
- If the answer is not in the CONTEXT, say exactly:
  "I don't have this information in the provided documents."
- Keep your answer under 3 sentences.
- If a specific number or rule is in CONTEXT, include it.

QUESTION: {question}

ANSWER:
""")

chain = prompt |llm | StrOutputParser()

In [10]:
DOCS = [
    "Gold loan foreclosure attracts 2% penalty if closed within 6 months of disbursal. No penalty after 6 months.",
    "Personal loan EMI is auto-debited on the 5th of every month. Insufficient funds attract a late fee of Rs 1000.",
    "Customers with CIBIL score below 650 are ineligible for a new personal loan or top-up loan.",
    "Loan renewal is available for Tier 1 and Tier 2 city customers with repayment score above 700 and no defaults in last 12 months.",
    "Gold loan renewal maximum is Rs 1,00,000 for salaried customers and Rs 50,000 for self-employed customers.",
]

In [ ]:
output = chain.invoke({
    "context": "\n".join(DOCS),
    "question": "What is the penalty for gold loan foreclosure?"})

In [12]:
output

'Gold loan foreclosure attracts a 2% penalty if closed within 6 months of disbursal. No penalty applies after 6 months.'

In [14]:
output

"I don't have this information in the provided documents."

In [26]:
DOCS = [
    "Gold loan foreclosure attracts 2% penalty if closed within 6 months of disbursal. No penalty after 6 months.",
    "Personal loan EMI is auto-debited on the 5th of every month. Insufficient funds attract a late fee of Rs 1000.",
    "Customers with CIBIL score below 650 are ineligible for a new personal loan or top-up loan.",
    "Loan renewal is available for Tier 1 and Tier 2 city customers with repayment score above 700 and no defaults in last 12 months.",
    "Gold loan renewal maximum is Rs 1,00,000 for salaried customers and Rs 50,000 for self-employed customers.",
    "Home loan renewal maximum is Rs 5,00,000 for salaried customers and Rs 50,000 for self-employed customers."
]


questions = "what is home loan foreclosure penalty?"



In [27]:
# keyword based searching
query_word = set(questions.lower().split())
query_word

{'foreclosure', 'home', 'is', 'loan', 'penalty?', 'what'}

In [21]:
word_1_chunk  = set(DOCS[2].lower().split())
word_1_chunk.intersection(query_word)

{'loan'}

In [28]:
result = []
for i in DOCS:
    doc_word = set(i.lower().split())
    overlap = doc_word.intersection(query_word)
    score = len(overlap)
    result.append({
        "score": score,
        "text": i
    })

In [29]:
result

[{'score': 2,
  'text': 'Gold loan foreclosure attracts 2% penalty if closed within 6 months of disbursal. No penalty after 6 months.'},
 {'score': 2,
  'text': 'Personal loan EMI is auto-debited on the 5th of every month. Insufficient funds attract a late fee of Rs 1000.'},
 {'score': 1,
  'text': 'Customers with CIBIL score below 650 are ineligible for a new personal loan or top-up loan.'},
 {'score': 2,
  'text': 'Loan renewal is available for Tier 1 and Tier 2 city customers with repayment score above 700 and no defaults in last 12 months.'},
 {'score': 2,
  'text': 'Gold loan renewal maximum is Rs 1,00,000 for salaried customers and Rs 50,000 for self-employed customers.'},
 {'score': 3,
  'text': 'Home loan renewal maximum is Rs 5,00,000 for salaried customers and Rs 50,000 for self-employed customers.'}]

In [30]:
result.sort(key=lambda x: x["score"], reverse=True)

In [31]:
result

[{'score': 3,
  'text': 'Home loan renewal maximum is Rs 5,00,000 for salaried customers and Rs 50,000 for self-employed customers.'},
 {'score': 2,
  'text': 'Gold loan foreclosure attracts 2% penalty if closed within 6 months of disbursal. No penalty after 6 months.'},
 {'score': 2,
  'text': 'Personal loan EMI is auto-debited on the 5th of every month. Insufficient funds attract a late fee of Rs 1000.'},
 {'score': 2,
  'text': 'Loan renewal is available for Tier 1 and Tier 2 city customers with repayment score above 700 and no defaults in last 12 months.'},
 {'score': 2,
  'text': 'Gold loan renewal maximum is Rs 1,00,000 for salaried customers and Rs 50,000 for self-employed customers.'},
 {'score': 1,
  'text': 'Customers with CIBIL score below 650 are ineligible for a new personal loan or top-up loan.'}]

In [32]:
def keyword_search(query, top_k=3):
    query_word = set(query.lower().split())
    result = []
    for i in DOCS:
        doc_word = set(i.lower().split())
        overlap = doc_word.intersection(query_word)
        score = len(overlap)
        result.append({
            "score": score,
            "text": i,
        })
    result.sort(key=lambda x: x["score"], reverse=True)
    # make sure score should be more then 1
    result = [i for i in result if i["score"] > 1]
    return result[:top_k]

In [35]:
query_a = "golds closure penalty"
keyword_search(query_a)

[]

In [34]:
query_b = "What  capital  India"

keyword_search(query_b)

[]

In [41]:
query_c = "Hi I want to close my loan emi"

docs = keyword_search(query_c)
docs

[{'score': 2,
  'text': 'Personal loan EMI is auto-debited on the 5th of every month. Insufficient funds attract a late fee of Rs 1000.'}]

In [47]:
output = chain.invoke({
    "context": '',
    "question": "What is the penalty for personal loan foreclosure?"})

In [48]:
output

"I don't have this information in the provided documents."

In [44]:
docs[0]['text']

'Personal loan EMI is auto-debited on the 5th of every month. Insufficient funds attract a late fee of Rs 1000.'